# INT234: Predictive Analytics Project Pipeline
## Predictive Modeling and Demographic Apparel Classification in Fast-Fashion E-Commerce (Zara Case Study)
**Author:** Deepak Rao  
**Course:** INT234 (Predictive Analytics)  
**Institution:** Lovely Professional University (LPU)


### 1. Import Required Libraries


In [ ]:
import os
import re
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

sns.set_theme(style="whitegrid")
%matplotlib inline


### 2. Load and Clean Master Dataset


In [ ]:
df = pd.read_csv("fast_fashion_zara_dataset.csv")
df = df.dropna(subset=["Price_INR", "Gender", "Category"]).copy()
print(f"Total records loaded: {len(df)}")
df.head()


### 3. Exploratory Data Analysis (EDA)


In [ ]:
# Distribution of Price by Gender
plt.figure(figsize=(9, 5))
sns.histplot(data=df, x="Price_INR", hue="Gender", bins=30, kde=True, palette={"Men": "#1f77b4", "Women": "#e377c2"})
plt.title("Price Distribution by Gender (Zara Fast Fashion)", fontsize=14, fontweight="bold")
plt.xlabel("Price in INR (₹)")
plt.ylabel("Product Count")
plt.show()

# Top 10 Product Categories
plt.figure(figsize=(10, 6))
top_cats = df["Category"].value_counts().head(10)
sns.barplot(x=top_cats.values, y=top_cats.index, hue=top_cats.index, palette="viridis", legend=False)
plt.title("Top 10 Fast Fashion Product Categories", fontsize=14, fontweight="bold")
plt.xlabel("Number of Items")
plt.ylabel("Category")
plt.show()


### 4. Train and Evaluate 5 Machine Learning Models across 80:20 and 75:25 Splits


In [ ]:
feature_cols = ["Category", "Price_INR", "Details_Length", "Num_Images",
                "Fabric_Cotton", "Fabric_Leather", "Fabric_Linen", "Fabric_Denim", "Fabric_Wool_Knit"]
X = df[feature_cols]
y = (df["Gender"] == "Women").astype(int)

cat_cols = ["Category"]
num_cols = ["Price_INR", "Details_Length", "Num_Images", "Fabric_Cotton", "Fabric_Leather", "Fabric_Linen", "Fabric_Denim", "Fabric_Wool_Knit"]

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Support Vector Machine": SVC(kernel="rbf", C=1.0, random_state=42)
}

results = []
for split_label, test_ratio in [("80:20 Split", 0.20), ("75:25 Split", 0.25)]:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_ratio, random_state=42, stratify=y)
    preprocessor = ColumnTransformer(transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ])
    X_train_proc = preprocessor.fit_transform(X_train)
    X_test_proc = preprocessor.transform(X_test)
    
    print("=" * 75)
    print(f"{split_label.upper()} RESULTS (Train: {len(X_train)}, Test: {len(X_test)})")
    print("=" * 75)
    for name, clf in models.items():
        clf.fit(X_train_proc, y_train)
        preds = clf.predict(X_test_proc)
        acc = accuracy_score(y_test, preds)
        prec = precision_score(y_test, preds, zero_division=0)
        rec = recall_score(y_test, preds, zero_division=0)
        f1 = f1_score(y_test, preds, zero_division=0)
        results.append({"Split": split_label, "Algorithm": name, "Accuracy": acc, "Precision": prec, "Recall": rec, "F1-Score": f1})
        print(f"{name:<25} | Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f}")

res_df = pd.DataFrame(results)
display(res_df)


### 5. Comparative Evaluation Visualizations


In [ ]:
# Accuracy comparison barplot
plt.figure(figsize=(11, 5))
sns.barplot(data=res_df, x="Algorithm", y="Accuracy", hue="Split", palette="Set2")
plt.title("Classification Accuracy Across 5 ML Models", fontsize=14, fontweight="bold")
plt.ylim(0.7, 1.0)
plt.xticks(rotation=15)
plt.show()
